In [ ]:
import cv2
import numpy as np

def create_short_fat_scratch_overlay(h, w, seed=42):
    rng = np.random.default_rng(seed)

    overlay = np.zeros((h, w, 4), dtype=np.uint8)

    x_max = int(w * 0.45)
    y_min = int(h * 0.55)
    scratches = np.zeros((h, w), dtype=np.float32)

    start_x = float(rng.integers(w//20, int(x_max * 0.8)))
    start_y = float(rng.integers(int(y_min * 1.1), int(h * 0.95)))

    length = float(rng.integers(w//20, w//10))

    base_angle = rng.uniform(-1.2, -0.4)

    num_strands = 6

    for strand in range(num_strands):
        x = start_x + rng.uniform(-4, 4)
        y = start_y + rng.uniform(-4, 4)

        angle = base_angle

        if strand <= 1:
            thickness = int(rng.integers(4, 6))
            intensity = 255.0
        else:
            thickness = int(rng.integers(1, 3))
            intensity = rng.uniform(150, 220)

        num_segments = int(rng.integers(8, 15))
        segment_len = length / num_segments

        for _ in range(num_segments):
            angle += rng.uniform(-0.15, 0.15)

            nx = x + segment_len * np.cos(angle)
            ny = y + segment_len * np.sin(angle)

            if rng.random() > 0.10:
                cv2.line(scratches, (int(x), int(y)), (int(nx), int(ny)), intensity, thickness)

            x, y = nx, ny

    scratches = np.clip(scratches, 0, 255).astype(np.uint8)
    scratches = cv2.GaussianBlur(scratches, (3,3), 0)
    final_alpha = scratches

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[..., 0] = 70   # R
    color[..., 1] = 68   # G
    color[..., 2] = 65   # B

    overlay[..., :3] = color
    overlay[..., 3] = final_alpha

    return overlay


Creates 1 Scratch Off-Target

In [ ]:
import os
import cv2
import numpy as np

def create_short_fat_scratch_overlay(h, w, seed=None):
    rng = np.random.default_rng(seed)

    overlay = np.zeros((h, w, 4), dtype=np.uint8)

    x_min = int(w * 0.55)
    y_max = int(h * 0.45)
    scratches = np.zeros((h, w), dtype=np.float32)
    start_x = float(rng.integers(int(x_min * 1.1), int(w * 0.95)))
    start_y = float(rng.integers(int(h * 0.05), int(y_max * 0.9)))
    length = float(rng.integers(w//20, w//10))
    base_angle = rng.uniform(2.0, 2.8)

    num_strands = 6

    for strand in range(num_strands):
        x = start_x + rng.uniform(-4, 4)
        y = start_y + rng.uniform(-4, 4)

        angle = base_angle
        if strand <= 1:
            thickness = int(rng.integers(4, 6))
            intensity = 255.0
        else:
            thickness = int(rng.integers(1, 3))
            intensity = rng.uniform(150, 220)

        num_segments = int(rng.integers(8, 15))
        segment_len = length / num_segments

        for _ in range(num_segments):
            angle += rng.uniform(-0.15, 0.15)

            nx = x + segment_len * np.cos(angle)
            ny = y + segment_len * np.sin(angle)

            if rng.random() > 0.10:
                cv2.line(scratches, (int(x), int(y)), (int(nx), int(ny)), intensity, thickness)

            x, y = nx, ny

    scratches = np.clip(scratches, 0, 255).astype(np.uint8)
    scratches = cv2.GaussianBlur(scratches, (3,3), 0)

    final_alpha = scratches

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[..., 0] = 70   # R
    color[..., 1] = 68   # G
    color[..., 2] = 65   # B

    overlay[..., :3] = color
    overlay[..., 3] = final_alpha

    return overlay


if __name__ == "__main__":
    input_dir = "/content/drive/MyDrive/dirtyImagesLevels/L0_cleanMasks"
    output_dir = "/content/drive/MyDrive/dirtyImagesLevels/L5_offtarget"

    os.makedirs(output_dir, exist_ok=True)

    valid_extensions = ('.png', '.jpg', '.jpeg')

    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(valid_extensions):
            continue

        input_path = os.path.join(input_dir, filename)

        output_filename = os.path.splitext(filename)[0] + ".png"
        output_path = os.path.join(output_dir, output_filename)
        original_img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

        if original_img is not None:
            h, w = original_img.shape[:2]

            damage_overlay = create_short_fat_scratch_overlay(h, w)

            damage_rgb = damage_overlay[..., :3]
            damage_alpha = damage_overlay[..., 3] / 255.0

            if original_img.shape[2] == 4:
                orig_bgr = original_img[..., :3].astype(float)
                orig_alpha = original_img[..., 3:4]
            else:
                orig_bgr = original_img.astype(float)
                orig_alpha = np.full((h, w, 1), 255, dtype=np.uint8)

            blended_bgr = orig_bgr.copy()
            for c in range(3):
                blended_bgr[..., c] = (damage_alpha * damage_rgb[..., c]) + ((1.0 - damage_alpha) * orig_bgr[..., c])

            blended_bgr = np.clip(blended_bgr, 0, 255).astype(np.uint8)

            final_output = np.concatenate([blended_bgr, orig_alpha], axis=-1)

            cv2.imwrite(output_path, final_output)
            print(f"Success! Saved {filename} -> {output_path}")
        else:
            print(f"Error: Could not read {input_path}")

Create 1 Scratch

In [ ]:
import os
import cv2
import numpy as np

def create_short_fat_scratch_overlay(h, w, seed=None):
    rng = np.random.default_rng(seed)

    overlay = np.zeros((h, w, 4), dtype=np.uint8)

    x_max = int(w * 0.45)
    y_min = int(h * 0.55)

    scratches = np.zeros((h, w), dtype=np.float32)

    start_x = float(rng.integers(w//20, int(x_max * 0.8)))
    start_y = float(rng.integers(int(y_min * 1.1), int(h * 0.95)))

    length = float(rng.integers(w//20, w//10))

    base_angle = rng.uniform(-1.2, -0.4)

    num_strands = 6

    for strand in range(num_strands):
        x = start_x + rng.uniform(-4, 4)
        y = start_y + rng.uniform(-4, 4)

        angle = base_angle

        if strand <= 1:
            thickness = int(rng.integers(4, 6))
            intensity = 255.0
        else:
            thickness = int(rng.integers(1, 3))
            intensity = rng.uniform(150, 220)

        num_segments = int(rng.integers(8, 15))
        segment_len = length / num_segments

        for _ in range(num_segments):
            angle += rng.uniform(-0.15, 0.15)

            nx = x + segment_len * np.cos(angle)
            ny = y + segment_len * np.sin(angle)

            if rng.random() > 0.10:
                cv2.line(scratches, (int(x), int(y)), (int(nx), int(ny)), intensity, thickness)

            x, y = nx

    scratches = np.clip(scratches, 0, 255).astype(np.uint8)
    scratches = cv2.GaussianBlur(scratches, (3,3), 0)

    final_alpha = scratches

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[..., 0] = 70
    color[..., 1] = 68
    color[..., 2] = 65

    overlay[..., :3] = color
    overlay[..., 3] = final_alpha

    return overlay


if __name__ == "__main__":
    input_dir = "/content/drive/MyDrive/dirtyImagesLevels/L0_cleanMasks"
    output_dir = "/content/drive/MyDrive/dirtyImagesLevels/L1_1scratch"

    os.makedirs(output_dir, exist_ok=True)

    valid_extensions = ('.png', '.jpg', '.jpeg')

    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(valid_extensions):
            continue

        input_path = os.path.join(input_dir, filename)

        output_filename = os.path.splitext(filename)[0] + ".png"
        output_path = os.path.join(output_dir, output_filename)

        original_img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

        if original_img is not None:
            h, w = original_img.shape[:2]

            damage_overlay = create_short_fat_scratch_overlay(h, w)

            damage_rgb = damage_overlay[..., :3]
            damage_alpha = damage_overlay[..., 3] / 255.0

            if original_img.shape[2] == 4:
                orig_bgr = original_img[..., :3].astype(float)
                orig_alpha = original_img[..., 3:4]
            else:
                orig_bgr = original_img.astype(float)
                orig_alpha = np.full((h, w, 1), 255, dtype=np.uint8)

            blended_bgr = orig_bgr.copy()
            for c in range(3):
                blended_bgr[..., c] = (damage_alpha * damage_rgb[..., c]) + ((1.0 - damage_alpha) * orig_bgr[..., c])

            blended_bgr = np.clip(blended_bgr, 0, 255).astype(np.uint8)

            final_output = np.concatenate([blended_bgr, orig_alpha], axis=-1)

            cv2.imwrite(output_path, final_output)
            print(f"Success! Saved {filename} -> {output_path}")
        else:
            print(f"Error: Could not read {input_path}")

Create 2 Scratches

In [ ]:

import os
import cv2
import numpy as np

def create_short_fat_scratch_overlay(h, w, seed=None):
    rng = np.random.default_rng(seed)
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    scratches = np.zeros((h, w), dtype=np.float32)

    x_max = int(w * 0.45)
    y_min = int(h * 0.55)
    num_scratches = 2

    for _ in range(num_scratches):

        start_x = float(rng.integers(w//20, int(x_max * 0.8)))
        start_y = float(rng.integers(int(y_min * 1.1), int(h * 0.95)))

        length = float(rng.integers(w//15, w//6))
        base_angle = rng.uniform(-1.4, -0.2)
        num_strands = int(rng.integers(6, 10))

        for strand in range(num_strands):
            x = start_x + rng.uniform(-5, 5)
            y = start_y + rng.uniform(-5, 5)

            angle = base_angle

            if strand <= 1:
                thickness = int(rng.integers(4, 7))
                intensity = 255.0
            else:
                thickness = int(rng.integers(1, 3))
                intensity = rng.uniform(150, 220)

            num_segments = int(rng.integers(8, 15))
            segment_len = length / num_segments

            for _ in range(num_segments):
                angle += rng.uniform(-0.15, 0.15)

                nx = x + segment_len * np.cos(angle)
                ny = y + segment_len * np.sin(angle)
                if rng.random() > 0.10:
                    cv2.line(scratches, (int(x), int(y)), (int(nx), int(ny)), intensity, thickness)

                x, y = nx, ny

    scratches = np.clip(scratches, 0, 255).astype(np.uint8)
    scratches = cv2.GaussianBlur(scratches, (3,3), 0)
    final_alpha = scratches

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[..., 0] = 70   # R
    color[..., 1] = 68   # G
    color[..., 2] = 65   # B

    overlay[..., :3] = color
    overlay[..., 3] = final_alpha

    return overlay


if __name__ == "__main__":
    input_dir = "/content/drive/MyDrive/dirtyImagesLevels/L0_cleanMasks"
    output_dir = "/content/drive/MyDrive/dirtyImagesLevels/L2_2scratches"

    os.makedirs(output_dir, exist_ok=True)

    valid_extensions = ('.png', '.jpg', '.jpeg')

    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(valid_extensions):
            continue

        input_path = os.path.join(input_dir, filename)

        output_filename = os.path.splitext(filename)[0] + ".png"
        output_path = os.path.join(output_dir, output_filename)
        original_img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

        if original_img is not None:
            h, w = original_img.shape[:2]
            damage_overlay = create_short_fat_scratch_overlay(h, w)

            damage_rgb = damage_overlay[..., :3]
            damage_alpha = damage_overlay[..., 3] / 255.0

            if original_img.shape[2] == 4:
                orig_bgr = original_img[..., :3].astype(float)
                orig_alpha = original_img[..., 3:4]
            else:
                orig_bgr = original_img.astype(float)
                orig_alpha = np.full((h, w, 1), 255, dtype=np.uint8)
            blended_bgr = orig_bgr.copy()
            for c in range(3):
                blended_bgr[..., c] = (damage_alpha * damage_rgb[..., c]) + ((1.0 - damage_alpha) * orig_bgr[..., c])

            blended_bgr = np.clip(blended_bgr, 0, 255).astype(np.uint8)
            final_output = np.concatenate([blended_bgr, orig_alpha], axis=-1)
            cv2.imwrite(output_path, final_output)
            print(f"Success! Saved {filename} -> {output_path}")
        else:
            print(f"Error: Could not read {input_path}")

Adds Speckles


In [ ]:
import os
import cv2
import numpy as np

def create_dust_overlay(h, w, density=0.001, seed=None):
    rng = np.random.default_rng(seed)

    overlay = np.zeros((h, w, 4), dtype=np.uint8)

    dust_mask = np.zeros((h, w), dtype=np.float32)

    num_dots = int(h * w * density)

    xs = rng.integers(0, w, size=num_dots)
    ys = rng.integers(0, h, size=num_dots)

    for i in range(num_dots):
        x, y = xs[i], ys[i]

        radius = int(rng.choice([2, 3, 4, 5], p=[0.40, 0.40, 0.15, 0.05]))

        intensity = rng.uniform(50, 255)

        cv2.circle(dust_mask, (x, y), radius, intensity, -1)

    dust_mask = cv2.GaussianBlur(dust_mask, (5, 5), 1.0)
    dust_mask = np.clip(dust_mask, 0, 255).astype(np.uint8)

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[..., 0] = 70
    color[..., 1] = 68
    color[..., 2] = 65

    overlay[..., :3] = color
    overlay[..., 3] = dust_mask

    return overlay


if __name__ == "__main__":
    input_dir = "/content/drive/MyDrive/dirtyImagesLevels/L0_cleanMasks"
    output_dir = "/content/drive/MyDrive/dirtyImagesLevels/L3_speckles"

    os.makedirs(output_dir, exist_ok=True)

    valid_extensions = ('.png', '.jpg', '.jpeg')

    for filename in os.listdir(input_dir):
        if not filename.lower().endswith(valid_extensions):
            continue

        input_path = os.path.join(input_dir, filename)

        output_filename = os.path.splitext(filename)[0] + ".png"
        output_path = os.path.join(output_dir, output_filename)

        original_img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)

        if original_img is not None:
            h, w = original_img.shape[:2]

            damage_overlay = create_dust_overlay(h, w, density=0.001)

            damage_rgb = damage_overlay[..., :3]
            damage_alpha = damage_overlay[..., 3] / 255.0

            if original_img.shape[2] == 4:
                orig_bgr = original_img[..., :3].astype(float)
                orig_alpha = original_img[..., 3:4]
            else:
                orig_bgr = original_img.astype(float)
                orig_alpha = np.full((h, w, 1), 255, dtype=np.uint8)

            blended_bgr = orig_bgr.copy()
            for c in range(3):
                blended_bgr[..., c] = (damage_alpha * damage_rgb[..., c]) + ((1.0 - damage_alpha) * orig_bgr[..., c])

            blended_bgr = np.clip(blended_bgr, 0, 255).astype(np.uint8)

            final_output = np.concatenate([blended_bgr, orig_alpha], axis=-1)

            cv2.imwrite(output_path, final_output)
            print(f"Success! Saved {filename} -> {output_path}")
        else:
            print(f"Error: Could not read {input_path}")